### 577. Employee Bonus
```
Table: Employee

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| empId       | int     |
| name        | varchar |
| supervisor  | int     |
| salary      | int     |
+-------------+---------+
empId is the column with unique values for this table.
Each row of this table indicates the name and the ID of an employee in addition to their salary and the id of their manager.
 

Table: Bonus

+-------------+------+
| Column Name | Type |
+-------------+------+
| empId       | int  |
| bonus       | int  |
+-------------+------+
empId is the column of unique values for this table.
empId is a foreign key (reference column) to empId from the Employee table.
Each row of this table contains the id of an employee and their respective bonus.
 

Write a solution to report the name and bonus amount of each employee who satisfies either of the following:

The employee has a bonus less than 1000.
The employee did not get any bonus.
Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Employee table:
+-------+--------+------------+--------+
| empId | name   | supervisor | salary |
+-------+--------+------------+--------+
| 3     | Brad   | null       | 4000   |
| 1     | John   | 3          | 1000   |
| 2     | Dan    | 3          | 2000   |
| 4     | Thomas | 3          | 4000   |
+-------+--------+------------+--------+
Bonus table:
+-------+-------+
| empId | bonus |
+-------+-------+
| 2     | 500   |
| 4     | 2000  |
+-------+-------+
Output: 
+------+-------+
| name | bonus |
+------+-------+
| Brad | null  |
| John | null  |
| Dan  | 500   |
+------+-------+

```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col

In [3]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [6]:
Data = [[3, 'Brad', None, 4000], [1, 'John', 3, 1000], [2, 'Dan', 3, 2000], [4, 'Thomas', 3, 4000]]
Schema=StructType([StructField("empid",IntegerType(),True),\
                  StructField("name",StringType(),True),\
                  StructField("supervisor",IntegerType(),True),\
                  StructField("salary",IntegerType(),True)]
                  
                 )
Employee=spark.createDataFrame(Data,Schema)
Employee.show()

+-----+------+----------+------+
|empid|  name|supervisor|salary|
+-----+------+----------+------+
|    3|  Brad|      NULL|  4000|
|    1|  John|         3|  1000|
|    2|   Dan|         3|  2000|
|    4|Thomas|         3|  4000|
+-----+------+----------+------+



In [4]:
data = [[2,500], [4,2000]]
schema=StructType([StructField("empid",IntegerType(),True),\
                  StructField("bonus",IntegerType(),True)]                
                 )
Bonus=spark.createDataFrame(data,schema)
Bonus.show()

+-----+-----+
|empid|bonus|
+-----+-----+
|    2|  500|
|    4| 2000|
+-----+-----+



In [9]:
result = (Employee.alias("E").
    join(Bonus.alias("B"),
         col('E.empid')==col('B.empid'),
         'left')
    .filter((col('B.bonus') < 1000) | (col('B.empid').isNull())
           ).select("E.name","B.bonus")
         )

In [10]:
result.show()

+----+-----+
|name|bonus|
+----+-----+
|Brad| NULL|
|John| NULL|
| Dan|  500|
+----+-----+



In [11]:
spark.stop()